# Linear_Regression Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Plant the truth.** Because the data is synthetic, we KNOW the hidden law (25 + 0.085·size) — perfect for checking whether the model recovers it.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 60

size_sqft = np.round(rng.uniform(700, 3000, n))
price_lakh = 25 + 0.085 * size_sqft + rng.normal(0, 15, n)

df = pd.DataFrame({"size_sqft": size_sqft, "price_lakh": price_lakh.round(1)})
print(df.head())
print("\nCorrelation(size, price):",
      df["size_sqft"].corr(df["price_lakh"]).round(3))

**2. Fit the line.** `coef_[0]` is the slope and `intercept_` the baseline — together they ARE the entire model, recovered from noisy data.

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["size_sqft"]]          # 2-D: (rows, ONE feature)
y = df["price_lakh"]

model = LinearRegression().fit(X, y)
w, b = model.coef_[0], model.intercept_

print(f"learned equation: price_hat = {w:.4f} * size + {b:.2f}")
print("planted truth    : price_hat = 0.0850 * size + 25.00")

**3. Price new flats.** Inference is one call — provided the input keeps the 2-D shape the model was trained on.

In [ ]:
import pandas as pd

new_flats = pd.DataFrame({"size_sqft": [950, 1600, 2600]})
new_flats["predicted_price_lakh"] = model.predict(new_flats).round(1)
print(new_flats.to_string(index=False))

# A bare 1-D list reads as 3 samples of 0 features. sklearn insists on
# (n_samples, n_features) - hence the DataFrame or [[950]]-style nesting.

## Part 2 — Practice

**4. Errors by hand.** Residuals tell the whole story; MAE averages the misses, MSE squares them, RMSE re-opens the units, R² compares against guessing the mean.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

model_s = LinearRegression().fit(X_train, y_train)
y_pred = model_s.predict(X_test)
residuals = y_test - y_pred

mae = np.abs(residuals).mean()
mse = (residuals ** 2).mean()
rmse = np.sqrt(mse)
r2 = 1 - (residuals ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()

print(f"MAE : {mae:.2f} lakh   <- typical miss")
print(f"MSE : {mse:.1f} lakh^2 <- squared units, outlier-brutal")
print(f"RMSE: {rmse:.2f} lakh  <- real units again")
print(f"R^2 : {r2:.3f}         <- share of variance explained")

**5. Let sklearn check your arithmetic.** Same numbers from the library confirm the formulas — trust, but verify your own maths first.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

checks = {
    "MAE":  (mae, mean_absolute_error(y_test, y_pred)),
    "MSE":  (mse, mean_squared_error(y_test, y_pred)),
    "RMSE": (rmse, mean_squared_error(y_test, y_pred) ** 0.5),
    "R^2":  (r2, r2_score(y_test, y_pred)),
}
for name, (mine, theirs) in checks.items():
    print(f"{name:<5} by hand: {mine:<10.4f} sklearn: {theirs:.4f}")

**6. Add a bedroom.** Multiple regression is the same engine: one weight per column, and `coef_` order matches the DataFrame's column order.

In [ ]:
rng2 = np.random.default_rng(42)
n2 = 120
size_sqft2 = np.round(rng2.uniform(700, 3000, n2))
bedrooms2 = rng2.integers(2, 6, n2)
price2 = 15 + 0.075 * size_sqft2 + 12 * bedrooms2 + rng2.normal(0, 15, n2)

X2 = pd.DataFrame({"size_sqft": size_sqft2, "bedrooms": bedrooms2})
y2 = pd.Series(price2, name="price_lakh")

model2 = LinearRegression().fit(X2, y2)
print(f"price_hat = {model2.intercept_:.2f}"
      f" + {model2.coef_[0]:.4f}*size + {model2.coef_[1]:.2f}*bedrooms")

dream_flat = pd.DataFrame({"size_sqft": [1400], "bedrooms": [3]})
print(f"a 1400 sqft, 3-bedroom flat -> {model2.predict(dream_flat)[0]:.1f} lakh")

**7. The extrapolation trap.** A line has no opinions — it extends forever. Flag inputs outside the training range instead of shipping fantasy prices.

In [ ]:
warehouse = pd.DataFrame({"size_sqft": [30000]})
print(f"predicted price for 30000 sqft: "
      f"{model.predict(warehouse)[0]:.1f} lakh")

# Nothing in the training data lived past 3000 sqft, yet the line happily
# continues its slope into fantasy territory. Guard inputs against the
# observed range, or retrain with data that covers it.

## Part 3 — Challenge

**8. When a line isn't enough.** PolynomialFeatures manufactures x², x³ columns so the same linear engine can draw a curve — degree 3 leaves degree 1 far behind.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(7)
x = rng.uniform(-3, 3, 70)
y_curve = 0.6 * x ** 2 - 1.5 * x + 2 + rng.normal(0, 0.8, 70)
X_curved = pd.DataFrame({"x": x})

for d in (1, 3):
    pipe = make_pipeline(PolynomialFeatures(degree=d),
                         LinearRegression()).fit(X_curved, y_curve)
    print(f"degree {d} R^2: {pipe.score(X_curved, y_curve):.3f}")

# The parabola in the data-generating formula means a straight line can
# never fit it - bending the FEATURES (not the engine) is the fix.

**9. Interrogate the residuals.** Left end too low, middle too high, right end too low — a U-shaped residual signature is the straight line admitting the data curves.

In [ ]:
deg1 = make_pipeline(PolynomialFeatures(degree=1),
                     LinearRegression()).fit(X_curved, y_curve)
residuals = y_curve - deg1.predict(X_curved)

order = np.argsort(x)
for i, chunk in enumerate(np.array_split(order, 3), start=1):
    print(f"group {i} (low/mid/high x): mean residual "
          f"{residuals[chunk].mean():.2f}")

# Healthy residuals scatter around zero with NO trend. This staircase
# pattern (negative-positive-negative) is curvature caught red-handed:
# the straight line was the wrong shape.